# Qwen2-VL Shared LoRA Sanity Check

Run this notebook once from top to bottom. It prepares the shared split if needed, evaluates zero-shot Qwen2-VL, sanity-checks fine-tuning labels, trains one shared LoRA adapter, evaluates the trained adapter on the same split, and writes a side-by-side comparison report.

This notebook intentionally does not run MoE, task-specific experts, GRPO, MoME, shared OCR experts, router training, or architecture changes.


## 1. Colab Repository Setup

Run this first in Colab. It clones or updates the repository and changes the working directory to the repo root.

In [ ]:
from pathlib import Path
from collections import Counter
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)
    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import re
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

required_packages = {
    "qwen-vl-utils": "qwen_vl_utils",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "Pillow": "PIL",
    "scikit-learn": "sklearn",
}
for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

import torch

# Colab may preinstall an old torchao version. Recent PEFT checks torchao when it
# exists, even though this notebook does not use torchao quantization.
try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

from peft import LoraConfig, get_peft_model
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

from src.data.answers import canonicalize_task_type, choose_training_answer
from src.evaluation.evaluator import build_prediction_records, summarize_quality_records_by_task

torch.__version__


## 3. Config

These settings drive the full one-pass run. Change limits here only if you want a smaller debug run before the real sanity check.


In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
LOCAL_CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/qwen2vl_lora_baseline_sample"
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl_lora_baseline_sample")
USE_GOOGLE_DRIVE_CHECKPOINT = True

MAX_SAMPLES = 5000
TRAIN_RATIO = 0.8
TEST_LIMIT = 1000
PRINT_LIMIT = 20
BATCH_SIZE = 2
EPOCHS = 3
LEARNING_RATE = 5e-5
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 42
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

DATA_LIMITS = {"docvqa": 1667, "chartqa": 1667, "textvqa": 1666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())
ZERO_SHOT_PREDICTIONS_PATH = PROJECT_ROOT / "outputs/predictions/qwen2vl_zero_shot_quality.jsonl"
SHARED_LORA_PREDICTIONS_PATH = PROJECT_ROOT / "outputs/predictions/qwen2vl_shared_lora_quality.jsonl"
COMPARISON_REPORT_PATH = PROJECT_ROOT / "outputs/predictions/qwen2vl_shared_lora_comparison.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
DEVICE


## 4. Checkpoint Storage

In [ ]:
CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 5. Prepare Data If Needed

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


def run_checked_command(command: list[str]) -> None:
    print("Running:", " ".join(command))
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise subprocess.CalledProcessError(
            result.returncode,
            command,
            output=result.stdout,
            stderr=result.stderr,
        )
    if result.stderr:
        print(result.stderr)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        run_checked_command(command)

    build_command = [sys.executable, str(PROJECT_ROOT / "scripts/build_multitask_data.py"), "--split", "validation"]
    run_checked_command(build_command)
else:
    print("Processed data already exists. Skipping data preparation.")


## 6. Load Records

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)


def canonical_task_type(record: dict) -> str:
    return canonicalize_task_type(record.get("task_type"), record.get("dataset")) or "textvqa"


all_records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        record["canonical_task_type"] = canonical_task_type(record)
        record["task_type"] = record["canonical_task_type"]
        record["chosen_training_answer"] = choose_training_answer(
            record.get("answers", []),
            record["canonical_task_type"],
        )
        all_records.append(record)

TASK_TYPES = ("chartqa", "docvqa", "textvqa")
all_records_by_task = {
    task_type: [record for record in all_records if record["canonical_task_type"] == task_type]
    for task_type in TASK_TYPES
}
for task_records in all_records_by_task.values():
    random.shuffle(task_records)

per_task_limit = max(1, MAX_SAMPLES // len(TASK_TYPES))
records = []
for task_type in TASK_TYPES:
    records.extend(all_records_by_task[task_type][:per_task_limit])

remaining_records = [
    record
    for task_records in all_records_by_task.values()
    for record in task_records[per_task_limit:]
]
random.shuffle(remaining_records)
records.extend(remaining_records[: max(0, MAX_SAMPLES - len(records))])
random.shuffle(records)

split_index = int(len(records) * TRAIN_RATIO)
train_records = records[:split_index]
test_records = records[split_index:]

print("Total selected records:", len(records))
print("Train records:", len(train_records))
print("Test records:", len(test_records))
print("Train records by task:", Counter(record["canonical_task_type"] for record in train_records))
print("Test records by task:", Counter(record["canonical_task_type"] for record in test_records))

records[0]


# One-Pass Run: Zero-Shot, Shared LoRA Train, Comparison


## 7. Load Base Model For Zero-Shot Evaluation


In [ ]:
def free_gpu_memory() -> None:
    for name in ["model", "optimizer", "scaler"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    model.to(DEVICE)
model.eval()

## 8. Prompting, Generation, and Prediction Helpers


In [ ]:
TASK_PROMPTS = {
    "chartqa": """Read the chart and answer the question.
Return only the requested value, label, or short phrase.
Do not explain.

Question: {question}
Answer:""",
    "docvqa": """Read the document and answer the question.
Return only the exact answer span.
Do not explain.

Question: {question}
Answer:""",
    "textvqa": """Read the visible text in the image and answer the question.
Return only the shortest answer.
Do not list all visible words.
Do not explain.

Question: {question}
Answer:""",
    "default": """Answer the question using the image.
Return only the shortest answer.
Do not explain.

Question: {question}
Answer:""",
}

TASK_GENERATION_KWARGS = {
    "chartqa": {"max_new_tokens": 8, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
    "docvqa": {"max_new_tokens": 16, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
    "textvqa": {"max_new_tokens": 12, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
    "default": {"max_new_tokens": 12, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
}


def format_question_prompt(question: str, task_type: str | None = None) -> str:
    canonical_task = canonicalize_task_type(task_type) or "default"
    template = TASK_PROMPTS.get(canonical_task, TASK_PROMPTS["default"])
    return template.format(question=question)


def generation_kwargs_for_task(task_type: str | None) -> dict:
    canonical_task = canonicalize_task_type(task_type) or "default"
    return TASK_GENERATION_KWARGS.get(canonical_task, TASK_GENERATION_KWARGS["default"])


def clean_generated_answer(answer: str) -> str:
    cleaned = " ".join(str(answer).strip().split())
    if not cleaned:
        return cleaned

    lowered = cleaned.lower()
    if lowered.startswith("unanswerable") or lowered.startswith("not a question"):
        return "unanswerable"

    tokens = cleaned.split()
    for start in range(1, len(tokens)):
        suffix = tokens[start:]
        if len(suffix) < 3:
            continue
        numeric_like = sum(bool(re.fullmatch(r"[\d.,:%$/-]+|[a-z]*\d+[a-z]*", token.lower())) for token in suffix)
        if numeric_like / len(suffix) >= 0.8:
            return " ".join(tokens[:start]).strip(" ,.;:")

    return cleaned.strip(" ,.;:")


def build_messages(image_path: str, question: str, task_type: str | None = None, answer: str | None = None):
    user_content = [
        {"type": "image", "image": str(PROJECT_ROOT / image_path)},
        {"type": "text", "text": format_question_prompt(question, task_type)},
    ]
    messages = [{"role": "user", "content": user_content}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def qwen_predict(example: dict, return_raw: bool = False):
    task_type = example.get("canonical_task_type") or example.get("task_type")
    messages = build_messages(example["image_path"], example["question"], task_type)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            **generation_kwargs_for_task(task_type),
        )

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_prediction = processor.batch_decode(generated_trimmed, skip_special_tokens=True)[0].strip()
    cleaned_prediction = clean_generated_answer(raw_prediction)
    if return_raw:
        return raw_prediction, cleaned_prediction
    return cleaned_prediction


## 9. Evaluate Zero-Shot Qwen2-VL


In [ ]:
zero_shot_records = []
zero_shot_subset = test_records[:TEST_LIMIT]

for index, example in enumerate(zero_shot_subset, start=1):
    raw_prediction, cleaned_prediction = qwen_predict(example, return_raw=True)
    zero_shot_records.append({
        "question": example["question"],
        "answers": example["answers"],
        "chosen_training_answer": example.get("chosen_training_answer"),
        "raw_prediction": raw_prediction,
        "cleaned_prediction": cleaned_prediction,
        "dataset": example["dataset"],
        "task_type": example["canonical_task_type"],
    })
    if index <= PRINT_LIMIT:
        print(f"[{index}/{len(zero_shot_subset)}] {example['dataset']} -> {cleaned_prediction!r}")

zero_shot_quality_records = build_prediction_records(
    [record["raw_prediction"] for record in zero_shot_records],
    zero_shot_subset,
    cleaned_predictions=[record["cleaned_prediction"] for record in zero_shot_records],
)
zero_shot_report = summarize_quality_records_by_task(zero_shot_quality_records)
ZERO_SHOT_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with ZERO_SHOT_PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
    for record in zero_shot_quality_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Zero-shot evaluated samples:", len(zero_shot_records))
print(json.dumps(zero_shot_report, indent=2))
print(f"Saved zero-shot per-sample quality records to {ZERO_SHOT_PREDICTIONS_PATH}")

zero_shot_quality_records[:3]


## 10. Build Shared-LoRA Fine-Tuning Dataset


## 10. Dataset and Collate Function

In [ ]:
class QwenVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        task_type = example["canonical_task_type"]
        answer = choose_training_answer(example.get("answers", []), task_type)
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answers": example.get("answers", []),
            "answer": answer,
            "task_type": task_type,
        }


def find_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    for start in range(len(sequence) - len(subsequence) + 1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1


def mask_prompt_tokens(labels, full_inputs, prompt_inputs, row_index: int) -> None:
    full_ids = full_inputs["input_ids"][row_index].tolist()
    prompt_ids = prompt_inputs["input_ids"][row_index][prompt_inputs["attention_mask"][row_index].bool()].tolist()
    prompt_start = find_subsequence(full_ids, prompt_ids)
    if prompt_start < 0:
        raise ValueError(f"Could not align prompt tokens for row {row_index}; refusing to train on unsafe labels.")
    labels[row_index, prompt_start:prompt_start + len(prompt_ids)] = -100


def collate_fn(batch):
    full_messages = [build_messages(item["image_path"], item["question"], item["task_type"], item["answer"]) for item in batch]
    prompt_messages = [build_messages(item["image_path"], item["question"], item["task_type"]) for item in batch]
    task_types = [item["task_type"] for item in batch]
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]
    reference_answers = [item["answers"] for item in batch]

    full_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in full_messages]
    prompt_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in prompt_messages]
    image_inputs, video_inputs = process_vision_info(full_messages)

    inputs = processor(
        text=full_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    prompt_inputs = processor(
        text=prompt_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()

    for row_index in range(labels.shape[0]):
        mask_prompt_tokens(labels, inputs, prompt_inputs, row_index)

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100

    special_token_ids = set(processor.tokenizer.all_special_ids)
    for token_id in special_token_ids:
        labels[labels == token_id] = -100
    inputs["labels"] = labels
    inputs["task_types"] = task_types
    inputs["questions"] = questions
    inputs["chosen_training_answers"] = answers
    inputs["reference_answers"] = reference_answers
    return inputs


train_dataset = QwenVQAFinetuneDataset(train_records)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
len(train_dataset)


## 11. Decode Labels Before Training

This cell must pass before training starts. It decodes the first batch labels after replacing `-100` with the pad token and asserts that labels contain only the assistant answer.


In [ ]:
def debug_decode_labels(processor, batch, n=4):
    labels = batch["labels"].detach().cpu().clone()

    decoded_rows = []
    for i in range(min(n, labels.shape[0])):
        row = labels[i]
        active_label_ids = row[row != -100]
        decoded = processor.tokenizer.decode(active_label_ids, skip_special_tokens=False)
        expected = batch["chosen_training_answers"][i]
        question = batch["questions"][i]
        print(f"[raw question {i}] {question!r}")
        print(f"[chosen training answer {i}] {expected!r}")
        print(f"[decoded labels {i}] {decoded!r}")
        decoded_rows.append(decoded)

        compact_decoded = " ".join(decoded.split())
        assert expected in decoded, f"Expected answer missing from decoded labels row {i}: {expected!r}"
        assert question not in decoded, f"Question leaked into labels row {i}."
        forbidden_fragments = ["<|im_start|>", "<|im_end|>", "<|vision_start|>", "<|vision_end|>", "<|image_pad|>", "assistant"]
        leaked_tokens = [token for token in forbidden_fragments if token in decoded]
        assert not leaked_tokens, f"Special/image/chat tokens leaked into labels row {i}: {leaked_tokens}"
        assert compact_decoded == expected, f"Labels should contain only the assistant answer for row {i}: {compact_decoded!r} != {expected!r}"
    return decoded_rows


debug_batch = next(iter(train_loader))
for key, value in debug_batch.items():
    if hasattr(value, "shape"):
        print(key, value.shape, value.dtype)
    else:
        print(key, value)

valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
print("label min/max:", int(valid_labels.min()), int(valid_labels.max()))
print("tokenizer vocab size:", processor.tokenizer.vocab_size)
print("tokenizer length with added/special tokens:", len(processor.tokenizer))
assert int(valid_labels.min()) >= 0
assert int(valid_labels.max()) < len(processor.tokenizer)
decoded_label_rows = debug_decode_labels(processor, debug_batch, n=4)
print("Batch check passed.")


## 12. Attach One Shared LoRA Adapter


In [ ]:
free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
if DEVICE != "cuda":
    model.to(DEVICE)
model.train()
model.print_trainable_parameters()

optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE)

## 13. Train Shared LoRA


In [ ]:
if DEVICE == "cuda":
    torch.cuda.empty_cache()

autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
loss_history = []
loss_by_task = {"chartqa": [], "docvqa": [], "textvqa": []}
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader, start=1):
        task_types = batch.pop("task_types")
        batch.pop("questions", None)
        batch.pop("chosen_training_answers", None)
        batch.pop("reference_answers", None)
        batch = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in batch.items()}

        with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)
        for task_type in task_types:
            loss_by_task.setdefault(task_type, []).append(loss_value)

        if step % 10 == 0 or step == 1:
            print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")


def mean_loss(losses: list[float]) -> float | None:
    if not losses:
        return None
    return sum(losses) / len(losses)


loss_summary = {
    "chartqa_loss": mean_loss(loss_by_task["chartqa"]),
    "docvqa_loss": mean_loss(loss_by_task["docvqa"]),
    "textvqa_loss": mean_loss(loss_by_task["textvqa"]),
    "overall_loss": mean_loss(loss_history),
}

print("Loss summary:")
for name, value in loss_summary.items():
    print(f"{name}: {value:.4f}" if value is not None else f"{name}: None")

loss_summary


## 14. Save Shared LoRA Adapter Checkpoint


In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f"Saved Qwen2-VL LoRA checkpoint to {CHECKPOINT_DIR}")

## 15. Evaluate Shared LoRA On The Same Test Split


In [ ]:
model.eval()
fine_tuned_records = []
fine_tuned_subset = test_records[:TEST_LIMIT]

for index, example in enumerate(fine_tuned_subset, start=1):
    raw_prediction, cleaned_prediction = qwen_predict(example, return_raw=True)
    fine_tuned_records.append({
        "index": index,
        "question": example["question"],
        "answers": example["answers"],
        "chosen_training_answer": example.get("chosen_training_answer"),
        "raw_prediction": raw_prediction,
        "cleaned_prediction": cleaned_prediction,
        "true_task": example["canonical_task_type"],
    })

    if index <= PRINT_LIMIT:
        print(f"[{index}/{len(fine_tuned_subset)}] {example['dataset']} -> {cleaned_prediction!r}")
        print("answers:", example["answers"])

shared_lora_quality_records = build_prediction_records(
    [record["raw_prediction"] for record in fine_tuned_records],
    fine_tuned_subset,
    cleaned_predictions=[record["cleaned_prediction"] for record in fine_tuned_records],
)
shared_lora_report = summarize_quality_records_by_task(shared_lora_quality_records)
SHARED_LORA_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with SHARED_LORA_PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
    for record in shared_lora_quality_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Shared LoRA evaluated samples:", len(fine_tuned_records))
print(json.dumps(shared_lora_report, indent=2))
print(f"Saved shared-LoRA per-sample quality records to {SHARED_LORA_PREDICTIONS_PATH}")
shared_lora_quality_records[:3]


## 16. Compare Zero-Shot vs Shared LoRA

Use this final report as the sanity gate. Shared LoRA should not collapse badly versus zero-shot before any task-specific LoRA or router/MoE experiment.


In [ ]:
def metric_delta(shared_report: dict, zero_report: dict, metric: str) -> float:
    return shared_report["overall"].get(metric, 0.0) - zero_report["overall"].get(metric, 0.0)


comparison_report = {
    "zero_shot": zero_shot_report,
    "shared_lora": shared_lora_report,
    "delta_shared_lora_minus_zero_shot": {
        "raw_exact_match": metric_delta(shared_lora_report, zero_shot_report, "raw_exact_match"),
        "normalized_exact_match": metric_delta(shared_lora_report, zero_shot_report, "normalized_exact_match"),
        "token_f1": metric_delta(shared_lora_report, zero_shot_report, "token_f1"),
        "containment": metric_delta(shared_lora_report, zero_shot_report, "containment"),
        "avg_output_token_length": metric_delta(shared_lora_report, zero_shot_report, "avg_output_token_length"),
        "pct_outputs_gt_10_tokens": metric_delta(shared_lora_report, zero_shot_report, "pct_outputs_gt_10_tokens"),
        "pct_outputs_gt_20_tokens": metric_delta(shared_lora_report, zero_shot_report, "pct_outputs_gt_20_tokens"),
    },
    "paths": {
        "zero_shot_predictions": str(ZERO_SHOT_PREDICTIONS_PATH),
        "shared_lora_predictions": str(SHARED_LORA_PREDICTIONS_PATH),
        "shared_lora_checkpoint": str(CHECKPOINT_DIR),
    },
}

COMPARISON_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_REPORT_PATH.write_text(json.dumps(comparison_report, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(comparison_report["delta_shared_lora_minus_zero_shot"], indent=2))
print(f"Saved comparison report to {COMPARISON_REPORT_PATH}")
comparison_report
